All Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

df = pd.read_csv("../data/credit_card_default.csv")

EDA Here

In [ ]:
df.head()
df.info()
df.describe()
df = df.drop(columns=['ID'])

Target distribution

In [ ]:
sns.countplot(x='default.payment.next.month', data=df)
plt.title("Default vs Non-Default Distribution")
plt.savefig("../outputs/figures/target_distribution.png")
plt.show()

Histogram of Credit Limit

In [ ]:
plt.hist(df['LIMIT_BAL'], bins=30)
plt.title("Credit Limit Distribution")
plt.xlabel("LIMIT_BAL")
plt.ylabel("Frequency")
plt.savefig("../outputs/figures/credit_limit_histogram.png")
plt.show()

Correlation heatmap

In [ ]:
plt.figure(figsize=(12,10))
sns.heatmap(df.corr(), cmap='coolwarm')
plt.title("Correlation Heatmap")
plt.savefig("../outputs/figures/heatmap.png")
plt.show()

Age Distribution

In [ ]:
plt.hist(df['AGE'], bins=30)
plt.title("Age Distribution")
plt.savefig("../outputs/figures/age_distribution.png")
plt.show()

Data Preparation

In [ ]:
X = df.drop(columns=['default.payment.next.month'])
y = df['default.payment.next.month']
#We are predicitng Y here

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Train Models

In [ ]:
#Logistic Regression Training
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

#Random Forest Training
rf = RandomForestClassifier(random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

Evaluation 

In [ ]:
#Logistic Regression Performance
print("Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))
print("F1 Score:", f1_score(y_test, y_pred_lr))
print("LR ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1]))

#Random Forest Performance
print("\nRandom Forest Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall:", recall_score(y_test, y_pred_rf))
print("F1 Score:", f1_score(y_test, y_pred_rf))
print("RF ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

# Create results string
results = f"""
Logistic Regression Performance:
Accuracy: {accuracy_score(y_test, y_pred_lr)}
Precision: {precision_score(y_test, y_pred_lr)}
Recall: {recall_score(y_test, y_pred_lr)}
F1 Score: {f1_score(y_test, y_pred_lr)}
ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1])}

Random Forest Performance:
Accuracy: {accuracy_score(y_test, y_pred_rf)}
Precision: {precision_score(y_test, y_pred_rf)}
Recall: {recall_score(y_test, y_pred_rf)}
F1 Score: {f1_score(y_test, y_pred_rf)}
ROC-AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])}
"""

# Save to file
with open("../outputs/model_results.txt", "w") as f:
    f.write(results)

print("Results saved successfully!")

Confusion Matrix

In [ ]:
ConfusionMatrixDisplay.from_estimator(lr, X_test_scaled, y_test)
plt.title("Logistic Regression Confusion Matrix")
plt.savefig("../outputs/figures/logistic_regression_confusion_matrix.png")
plt.show()

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test)
plt.title("Random Forest Confusion Matrix")
plt.savefig("../outputs/figures/random_forest_confusion_matrix.png")
plt.show()

Logistic Regression Coefficients

In [ ]:
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr.coef_[0]
})
coef_df = coef_df.sort_values(by='Coefficient')
coef_df.head(10)
coef_df.tail(10)
plt.figure(figsize=(10, 8))
plt.barh(coef_df['Feature'], coef_df['Coefficient'])
plt.title("Logistic Regression Feature Coefficients")
plt.xlabel("Coefficient Value")
plt.ylabel("Feature")
plt.axvline(0, color='black', linewidth=1)
plt.savefig("../outputs/figures/logistic_regression_feature_importance.png")
plt.show()

Feature Importance for Random Tree

In [ ]:
importances = rf.feature_importances_
feature_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': importances
})
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False)
feature_importance_df.head(10)
top_features = feature_importance_df.head(10)
plt.figure()
plt.barh(top_features['Feature'], top_features['Importance'])
plt.title("Top 10 Feature Importances (Random Forest)")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.gca().invert_yaxis()
plt.savefig("../outputs/figures/random_tree_feature_importance.png")
plt.show()